In [1]:
# -------------------------------------------------------------------------------------------------
# 0. INSTALLATION & SETUP
# -------------------------------------------------------------------------------------------------
!pip -q install -U "pandas<2.4.0,>=2.0" "scikit-learn" "scipy" "transformer-lens"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 9.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 6.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 73.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 103.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 90.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 5.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 2.3.3 which is incompatible.


In [2]:
# =================================================================================================
# 🔬 MILESTONE 19.5-R10
# ORTHOGONAL RESIDUAL ARCHITECTURE & TEMPORAL CAUSAL BENCHMARK
#
# Version : M19.5-R10.0
# Model   : Qwen/Qwen2.5-0.5B
# Backend : TransformerLens / HookedTransformer
# =================================================================================================

import os
import json
import math
import time
import random
import hashlib
import warnings

import numpy as np
import pandas as pd
import torch

from sklearn.linear_model import RidgeCV
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error

warnings.filterwarnings("ignore")
from transformer_lens import HookedTransformer

# -------------------------------------------------------------------------------------------------
# 1. CONFIGURATION
# -------------------------------------------------------------------------------------------------

SEED = 195926
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "Qwen/Qwen2.5-0.5B"

N_TRAIN = 160
N_CALIBRATION = 80
N_CLEAN_TEST = 80
N_FACTORIAL_STEP = 100
CHANGEPOINT_STEP = 35  # For Regime C temporal causality test
PREFERRED_LAYER = 6
EPS = 1e-8

print("=" * 110)
print("🔬 MILESTONE 19.5-R10 — ORTHOGONAL RESIDUAL ARCHITECTURE BENCHMARK")
print("=" * 110)
print(f"Seed: {SEED} | Device: {DEVICE} | Model: {MODEL_NAME}")

# -------------------------------------------------------------------------------------------------
# 2. AUDIT / LINEAGE LOGGER
# -------------------------------------------------------------------------------------------------

class LineageLogger:
    def __init__(self):
        self.events = []
        self.counter = 0

    def log(self, event_type, payload=None):
        self.counter += 1
        self.events.append({
            "id": self.counter,
            "timestamp": time.time(),
            "event": event_type,
            "payload": payload or {},
        })

LOGGER = LineageLogger()

# -------------------------------------------------------------------------------------------------
# 3. 2x2 FACTORIAL DATASET GENERATION (WITH CHANGEPOINT DYNAMICS)
# -------------------------------------------------------------------------------------------------

def content_fingerprint(row):
    raw = f"{row['a']:.8f}|{row['b']:.8f}|{row['mechanism']:.8f}|{row['context_type']}|{row['split']}|{row['idx']}"
    return hashlib.sha256(raw.encode()).hexdigest()

def generate_sample(split, idx, regime="NOMINAL_A", t_step=None):
    context_type = "STANDARD"

    if regime in ["TRAIN", "CALIBRATION", "CLEAN_TEST", "NOMINAL_A"]:
        a = np.random.uniform(-1.0, 1.0)
        b = np.random.uniform(-1.0, 1.0)
        mechanism = 0.8 * a - 0.55 * b

    elif regime == "NOVEL_VALID_B":
        # Format Novelty: Representation manifold shifted, but supports within domain
        a = np.random.uniform(-1.0, 1.0)
        b = np.random.uniform(-1.0, 1.0)
        context_type = "NOVEL_FORMAT"
        mechanism = 0.8 * a - 0.55 * b

    elif regime == "FAMILIAR_SHIFT_C":
        # Temporal Changepoint: Step 0..34 is Nominal, Step >= 35 is Causal-Shifted
        a = np.random.uniform(-1.0, 1.0)
        b = np.random.uniform(-1.0, 1.0)
        context_type = "STANDARD"
        if t_step is not None and t_step >= CHANGEPOINT_STEP:
            mechanism = -0.8 * a - 0.55 * b  # Causal anomaly
        else:
            mechanism = 0.8 * a - 0.55 * b   # Pre-shift nominal

    elif regime == "NOVEL_SHIFT_D":
        # Novel Format AND immediate Causal Shift
        a = np.random.uniform(-1.0, 1.0)
        b = np.random.uniform(-1.0, 1.0)
        context_type = "NOVEL_FORMAT"
        mechanism = -0.8 * a - 0.55 * b

    else:
        raise ValueError(regime)

    row = {
        "split": split,
        "idx": idx,
        "t_step": t_step,
        "a": float(a),
        "b": float(b),
        "regime": regime,
        "context_type": context_type,
        "mechanism": float(mechanism),
    }
    row["fingerprint"] = content_fingerprint(row)
    return row

train_data = [generate_sample("TRAIN", i, "TRAIN") for i in range(N_TRAIN)]
cal_data = [generate_sample("CALIBRATION", i, "CALIBRATION") for i in range(N_CALIBRATION)]
clean_test = [generate_sample("CLEAN_TEST", i, "CLEAN_TEST") for i in range(N_CLEAN_TEST)]

stream_A = [generate_sample("REGIME_A", i, "NOMINAL_A", t_step=i) for i in range(N_FACTORIAL_STEP)]
stream_B = [generate_sample("REGIME_B", i, "NOVEL_VALID_B", t_step=i) for i in range(N_FACTORIAL_STEP)]
stream_C = [generate_sample("REGIME_C", i, "FAMILIAR_SHIFT_C", t_step=i) for i in range(N_FACTORIAL_STEP)]
stream_D = [generate_sample("REGIME_D", i, "NOVEL_SHIFT_D", t_step=i) for i in range(N_FACTORIAL_STEP)]

# Collision Check
all_splits = [train_data, cal_data, clean_test, stream_A, stream_B, stream_C, stream_D]
fps = [set(x["fingerprint"] for x in s) for s in all_splits]
has_overlap = any(len(fps[i] & fps[j]) > 0 for i in range(len(fps)) for j in range(i + 1, len(fps)))
assert not has_overlap, "Collision detected across factorial partitions!"
print("🔐 Content & Split Leakage Guard: PASS")

# -------------------------------------------------------------------------------------------------
# 4. SENSOR & ACTIVATION EXTRACTION
# -------------------------------------------------------------------------------------------------

model = HookedTransformer.from_pretrained(MODEL_NAME, device=DEVICE)
model.eval()

hook_name = None
for name in model.hook_dict.keys():
    if f"blocks.{PREFERRED_LAYER}.hook_resid_post" in name:
        hook_name = name
        break
if not hook_name:
    hook_name = [k for k in model.hook_dict.keys() if "hook_resid_post" in k][0]

print(f"🎯 Sensor Hook: {hook_name}")

def make_prompt(row):
    if row["context_type"] == "NOVEL_FORMAT":
        return (
            "<<SYSTEM EXPERIMENTAL QUERY PROTOCOL>>\n"
            f"Coordinate Alpha: {row['a']:.6f}\n"
            f"Coordinate Beta:  {row['b']:.6f}\n"
            "Synthesize latent transformation outcome value."
        )
    return (
        "You are solving a numerical reasoning task.\n"
        f"Input A = {row['a']:.5f}\n"
        f"Input B = {row['b']:.5f}\n"
        "Infer the relationship between the two quantities.\n"
        "Provide the predicted resulting value."
    )

@torch.no_grad()
def extract_features(rows, batch_size=16):
    features = []
    for start in range(0, len(rows), batch_size):
        batch = rows[start:start + batch_size]
        prompts = [make_prompt(x) for x in batch]
        tokens = model.to_tokens(prompts, prepend_bos=True).to(DEVICE)
        _, cache = model.run_with_cache(tokens, names_filter=[hook_name], return_type=None)
        act = cache[hook_name].mean(dim=1).detach().float().cpu().numpy()
        features.append(act)
        del cache, tokens
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
    return np.concatenate(features, axis=0)

print("🔬 Extracting activations from sensor...")
train_X = extract_features(train_data)
cal_X = extract_features(cal_data)
clean_X = extract_features(clean_test)

A_X = extract_features(stream_A)
B_X = extract_features(stream_B)
C_X = extract_features(stream_C)
D_X = extract_features(stream_D)

train_y = np.array([x["mechanism"] for x in train_data])
cal_y = np.array([x["mechanism"] for x in cal_data])
clean_y = np.array([x["mechanism"] for x in clean_test])

A_y = np.array([x["mechanism"] for x in stream_A])
B_y = np.array([x["mechanism"] for x in stream_B])
C_y = np.array([x["mechanism"] for x in stream_C])
D_y = np.array([x["mechanism"] for x in stream_D])

# -------------------------------------------------------------------------------------------------
# 5. CORE PREDICTOR & NOVELTY SENSOR (AXIS 1)
# -------------------------------------------------------------------------------------------------

class OptimizedSelfModel:
    def __init__(self):
        self.scaler = StandardScaler()
        self.model = RidgeCV(alphas=np.logspace(-3, 4, 20))
        self.frozen = False

    def fit(self, X, y):
        if self.frozen:
            raise RuntimeError("SelfModel is frozen.")
        X_s = self.scaler.fit_transform(X)
        self.model.fit(X_s, y)
        LOGGER.log("SELF_MODEL_FIT", {"alpha": float(self.model.alpha_), "n": len(y)})

    def freeze(self):
        self.frozen = True
        LOGGER.log("SELF_MODEL_FROZEN")

    def predict(self, X):
        return self.model.predict(self.scaler.transform(X))

self_model = OptimizedSelfModel()
self_model.fit(train_X, train_y)
self_model.freeze()

class NoveltySensorAxis1:
    """Axis 1: Pure Geometric Novelty Sensor."""
    def __init__(self, n_components=16):
        self.pca = PCA(n_components=n_components)
        self.scaler = StandardScaler()
        self.mean_z = None
        self.cov_inv = None
        self.tau_novel = None

    def fit(self, X):
        X_s = self.scaler.fit_transform(X)
        Z = self.pca.fit_transform(X_s)
        self.mean_z = np.mean(Z, axis=0)
        cov = np.cov(Z, rowvar=False) + np.eye(Z.shape[1]) * 1e-4
        self.cov_inv = np.linalg.inv(cov)

        diff = Z - self.mean_z
        d_mah = np.sqrt(np.sum(diff @ self.cov_inv * diff, axis=1))
        self.tau_novel = float(np.quantile(d_mah, 0.95))

    def evaluate_novelty(self, X):
        X_s = self.scaler.transform(X)
        Z = self.pca.transform(X_s)
        diff = Z - self.mean_z
        d_mah = np.sqrt(np.sum(diff @ self.cov_inv * diff, axis=1))
        novelty_score = 1.0 / (1.0 + np.exp(-1.5 * (d_mah - self.tau_novel)))
        return np.clip(novelty_score, 0.01, 0.99)

novelty_sensor = NoveltySensorAxis1()
novelty_sensor.fit(train_X)

cal_preds = self_model.predict(cal_X)
cal_errors = np.abs(cal_y - cal_preds)
base_error_std = float(np.std(cal_errors)) + EPS
base_error_mean = float(np.mean(cal_errors))

# -------------------------------------------------------------------------------------------------
# 6. ORTHOGONAL RESIDUAL ENGINE (AXIS 2 & AXIS 3 DECOUPLED)
# -------------------------------------------------------------------------------------------------

class OrthogonalGovernanceEngine:
    """
    R10 Core Architecture:
    - Surprise Z-Score depends ONLY on behavioral error vs base_std (Zero Novelty Coupling).
    - Novelty drives Confidence moderations directly, completely bypassing CUSUM.
    """
    def __init__(self, base_mean, base_std, h_threshold=4.5):
        self.base_mean = base_mean
        self.base_std = base_std
        self.h = h_threshold
        self.reset()

    def reset(self):
        self.cusum = 0.0
        self.in_quarantine = False

    def update_history(self, recent_error):
        if recent_error is not None:
            # AXIS 2: Pure Behavioral Surprise (No novelty scale factor leakage!)
            surprise_z = (recent_error - self.base_mean) / self.base_std

            # AXIS 3: CUSUM Accumulator for Causal Violations
            if surprise_z > 2.0:
                self.cusum = max(0.0, 0.92 * self.cusum + (surprise_z - 2.0))
            else:
                self.cusum = max(0.0, 0.65 * self.cusum - 0.8)

            if not self.in_quarantine:
                if self.cusum >= self.h:
                    self.in_quarantine = True
            else:
                if self.cusum <= 0.6:
                    self.in_quarantine = False
                    self.cusum = 0.0

    def evaluate_governance(self, novelty):
        causal_state = float(1.0 / (1.0 + np.exp(-2.0 * (self.cusum - self.h))))

        # Orthogonal Decision Matrix
        if self.in_quarantine or causal_state >= 0.50:
            action = "QUARANTINE"
            conf = 0.05
        elif causal_state >= 0.25:
            action = "REDUCED_ACT"
            conf = 0.35
        elif novelty >= 0.60:
            # Regime B: Novel Geometry, Zero Causal Anomaly -> ACT with moderated confidence
            action = "ACT"
            conf = 0.60
        else:
            # Regime A: Nominal Familiar -> ACT with full confidence
            action = "ACT"
            conf = 0.95

        return float(novelty), causal_state, conf, action

engine_r10 = OrthogonalGovernanceEngine(base_error_mean, base_error_std)
LOGGER.log("ORTHOGONAL_ENGINE_INITIALIZED")

# -------------------------------------------------------------------------------------------------
# 7. FACTORIAL STREAM EXECUTION
# -------------------------------------------------------------------------------------------------

def run_stream(X_data, y_data):
    novelties = novelty_sensor.evaluate_novelty(X_data)
    preds = self_model.predict(X_data)
    errors = np.abs(y_data - preds)

    engine_r10.reset()
    acts, causal_states, confs = [], [], []

    for t in range(len(X_data)):
        nov = novelties[t]
        n_t, c_t, conf_t, act = engine_r10.evaluate_governance(nov)
        acts.append(act)
        causal_states.append(c_t)
        confs.append(conf_t)

        # Update historical evidence without passing novelty
        engine_r10.update_history(errors[t])

    return {
        "act_rate": float(np.mean(np.array(acts) == "ACT")),
        "quarantine_rate": float(np.mean(np.array(acts) == "QUARANTINE")),
        "mean_novelty": float(np.mean(novelties)),
        "mean_causal_evidence": float(np.mean(causal_states)),
        "mae": float(mean_absolute_error(y_data, preds)),
        "actions": acts,
    }

print("\n⚙️ Evaluating R10 Factorial Streams...")
res_A = run_stream(A_X, A_y)
res_B = run_stream(B_X, B_y)
res_C = run_stream(C_X, C_y)
res_D = run_stream(D_X, D_y)

# -------------------------------------------------------------------------------------------------
# 8. TEMPORAL ADVANTAGE & PERMUTATION TEST (CHANGEPOINT DISCRIMINATION)
# -------------------------------------------------------------------------------------------------

C_errors = np.abs(C_y - self_model.predict(C_X))
C_novs = novelty_sensor.evaluate_novelty(C_X)

# Compute Real Changepoint Latency and False Alarm Rate (t < CHANGEPOINT_STEP)
engine_r10.reset()
real_pre_alarms = 0
first_detect_step = None

for t in range(len(C_X)):
    _, _, _, act_t = engine_r10.evaluate_governance(C_novs[t])
    if t < CHANGEPOINT_STEP and act_t == "QUARANTINE":
        real_pre_alarms += 1
    if t >= CHANGEPOINT_STEP and act_t == "QUARANTINE" and first_detect_step is None:
        first_detect_step = t
    engine_r10.update_history(C_errors[t])

real_ttd = (first_detect_step - CHANGEPOINT_STEP) if first_detect_step is not None else 100
real_pre_alarm_rate = real_pre_alarms / CHANGEPOINT_STEP

# Monte Carlo Shuffled Streams: Destroys temporal locality
N_PERM = 50
shuff_pre_alarms = []
for p in range(N_PERM):
    shuffled_errs = list(C_errors)
    random.Random(p + 1000).shuffle(shuffled_errs)
    engine_r10.reset()
    pre_alarms_p = 0
    for t in range(len(C_X)):
        _, _, _, act_t = engine_r10.evaluate_governance(C_novs[t])
        if t < CHANGEPOINT_STEP and act_t == "QUARANTINE":
            pre_alarms_p += 1
        engine_r10.update_history(shuffled_errs[t])
    shuff_pre_alarms.append(pre_alarms_p / CHANGEPOINT_STEP)

mean_shuff_pre_alarm = float(np.mean(shuff_pre_alarms))

# -------------------------------------------------------------------------------------------------
# 9. SCIENTIFIC AUDIT CRITERIA (M19.5-R10)
# -------------------------------------------------------------------------------------------------

criteria = {
    # 1. Regime A: Pure Nominal Performance
    "REGIME_A_NOMINAL_ACT_PASS": bool(res_A["act_rate"] == 1.0 and res_A["quarantine_rate"] == 0.0),

    # 2. Regime B: Novel Geometry Decoupled from Quarantine
    "REGIME_B_NOVEL_VALID_DECOUPLED": bool(
        res_B["mean_novelty"] >= 0.70 and res_B["quarantine_rate"] == 0.0 and res_B["act_rate"] >= 0.90
    ),

    # 3. Regime C: Changepoint Shift Detected in Second Half
    "REGIME_C_FAMILIAR_SHIFT_DETECTED": bool(
        res_C["mean_novelty"] < 0.35 and res_C["quarantine_rate"] >= 0.55
    ),

    # 4. Regime D: Novel Geometry AND Shift Detected
    "REGIME_D_NOVEL_SHIFT_DETECTED": bool(
        res_D["mean_novelty"] >= 0.70 and res_D["quarantine_rate"] >= 0.85
    ),

    # 5. True Orthogonal Axis Separation
    "ORTHOGONAL_AXIS_SEPARATION": bool(
        abs(res_B["mean_novelty"] - res_C["mean_novelty"]) >= 0.45 and
        abs(res_B["mean_causal_evidence"] - res_D["mean_causal_evidence"]) >= 0.50
    ),

    # 6. Temporal Ordering Advantage: Shuffled sequence triggers massive false alarms in nominal window
    "TEMPORAL_CAUSAL_ORDER_ADVANTAGE": bool((mean_shuff_pre_alarm - real_pre_alarm_rate) >= 0.25),

    # 7. Audit Trail Complete
    "LINEAGE_AUDIT_PASS": bool(
        len({"SELF_MODEL_FIT", "SELF_MODEL_FROZEN", "ORTHOGONAL_ENGINE_INITIALIZED"} - set(x["event"] for x in LOGGER.events)) == 0
    )
}

# -------------------------------------------------------------------------------------------------
# 10. FINAL REPORT & TELEMETRY
# -------------------------------------------------------------------------------------------------

print("\n" + "=" * 110)
print("🏁 MILESTONE 19.5-R10 AUDIT RESULTS")
print("=" * 110)

passed = 0
for name, val in criteria.items():
    status = "✅ PASS" if val else "❌ FAIL"
    print(f"{status} | {name}")
    if val: passed += 1

print("-" * 110)
print(f"Criteria Passed: {passed}/{len(criteria)}")
overall_pass = bool(passed == len(criteria))
print(f"Overall M19.5-R10 Status: {'PASS' if overall_pass else 'FAIL'}")

print("\n" + "=" * 110)
print("📋 2x2 FACTORIAL ORTHOGONAL SEPARATION REPORT")
print("=" * 110)
print(f"{'Regime':<20} | {'Novelty (Axis 1)':<18} | {'Causal Evid (Axis 3)':<20} | {'ACT Rate':<10} | {'Quarantine':<12}")
print("-" * 90)
print(f"{'A: Nominal':<20} | {res_A['mean_novelty']:<18.4f} | {res_A['mean_causal_evidence']:<20.4f} | {res_A['act_rate']:<10.1%} | {res_A['quarantine_rate']:<12.1%}")
print(f"{'B: Novel-Valid':<20} | {res_B['mean_novelty']:<18.4f} | {res_B['mean_causal_evidence']:<20.4f} | {res_B['act_rate']:<10.1%} | {res_B['quarantine_rate']:<12.1%}")
print(f"{'C: Familiar-Shift':<20} | {res_C['mean_novelty']:<18.4f} | {res_C['mean_causal_evidence']:<20.4f} | {res_C['act_rate']:<10.1%} | {res_C['quarantine_rate']:<12.1%}")
print(f"{'D: Novel-Shift':<20} | {res_D['mean_novelty']:<18.4f} | {res_D['mean_causal_evidence']:<20.4f} | {res_D['act_rate']:<10.1%} | {res_D['quarantine_rate']:<12.1%}")

print(f"\nTemporal Ordering Audit on Regime C (Pre-Changepoint Step 0..{CHANGEPOINT_STEP-1}):")
print(f"  - Real Sequence False Alarm Rate  : {real_pre_alarm_rate:.1%} (TTD after shift: {real_ttd} steps)")
print(f"  - Shuffled Sequences Pre-Alarm Rate: {mean_shuff_pre_alarm:.1%}")
print(f"  - Temporal Discriminative Gap      : +{mean_shuff_pre_alarm - real_pre_alarm_rate:.1%}")

telemetry = {
    "milestone": "M19.5-R10",
    "overall_pass": bool(overall_pass),
    "criteria": {str(k): bool(v) for k, v in criteria.items()},
    "factorial_matrix": {
        "regime_A": res_A,
        "regime_B": res_B,
        "regime_C": res_C,
        "regime_D": res_D
    },
    "temporal_audit": {
        "real_pre_alarm_rate": real_pre_alarm_rate,
        "mean_shuff_pre_alarm": mean_shuff_pre_alarm,
        "detection_delay_steps": real_ttd
    }
}

OUTPUT_FILE = "m19_5_r10_final_audit_results.json"
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(telemetry, f, indent=2)

print(f"\n💾 Telemetry successfully written to {OUTPUT_FILE}")
print("=" * 110)

🔬 MILESTONE 19.5-R10 — ORTHOGONAL RESIDUAL ARCHITECTURE BENCHMARK
Seed: 195926 | Device: cuda | Model: Qwen/Qwen2.5-0.5B
🔐 Content & Split Leakage Guard: PASS


config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Loaded pretrained model Qwen/Qwen2.5-0.5B into HookedTransformer
🎯 Sensor Hook: blocks.6.hook_resid_post
🔬 Extracting activations from sensor...

⚙️ Evaluating R10 Factorial Streams...

🏁 MILESTONE 19.5-R10 AUDIT RESULTS
✅ PASS | REGIME_A_NOMINAL_ACT_PASS
❌ FAIL | REGIME_B_NOVEL_VALID_DECOUPLED
✅ PASS | REGIME_C_FAMILIAR_SHIFT_DETECTED
✅ PASS | REGIME_D_NOVEL_SHIFT_DETECTED
❌ FAIL | ORTHOGONAL_AXIS_SEPARATION
✅ PASS | TEMPORAL_CAUSAL_ORDER_ADVANTAGE
✅ PASS | LINEAGE_AUDIT_PASS
--------------------------------------------------------------------------------------------------------------
Criteria Passed: 5/7
Overall M19.5-R10 Status: FAIL

📋 2x2 FACTORIAL ORTHOGONAL SEPARATION REPORT
Regime               | Novelty (Axis 1)   | Causal Evid (Axis 3) | ACT Rate   | Quarantine  
------------------------------------------------------------------------------------------
A: Nominal           | 0.0757             | 0.0001               | 100.0%     | 0.0%        
B: Novel-Valid       | 0.9900   